# Dithering algorithm comparison

Migrated from `dither_test/`. The 6-colour ACeP palette can only show Black, White, Yellow,
Red, Blue, Green — so the dithering algorithm choice has a big effect on perceived image
quality. This notebook applies a curated set of error-diffusion and ordered-dithering
algorithms to a few real photos from Immich and shows them side-by-side, with timing.

Production uses Atkinson with perceptual weighting (`atkinson_weighted` is the closest
match — the actual production version is numba-JIT'd, equivalent to `atkinson_fast`). This
notebook is the place to evaluate alternatives if you want to switch.

Algorithm taxonomy:
- **Error diffusion** (`floyd_steinberg`, `atkinson`, `jarvis`, `stucki`, `sierra`,
  `sierra_lite`, `burkes`) — quantise pixels left-to-right, push the rounding error onto
  unprocessed neighbours.
- **Ordered** (`bayer4`, `bayer8`, `bayer4_strong`) — add a deterministic threshold pattern
  before quantising. No error spreading; pattern is independent of content.
- **PIL built-ins** (`pil_fs`, `pil_none`) — for reference.

In [ ]:
import sys
sys.path.insert(0, '.')
from _helpers import bootstrap, immich_client, fetch_pool, download_image, silenced, show_grid
bootstrap()

import time
import numpy as np
from PIL import Image
from waveshare_epd.epd7in3e import EPD_WIDTH, EPD_HEIGHT, _crop_center
from _dither import DITHER_ALGORITHMS, apply_dithering, PALETTE_RGB, PALETTE_NAMES

# Pure-Python algorithms run at ~30s per 800x480 image; keep a curated subset by default.
# Toggle SHOW_ALL = True to run everything (will take several minutes).
DEFAULT_ALGOS = ['atkinson_fast', 'atkinson', 'atkinson_weighted',
                 'floyd_steinberg', 'floyd_steinberg_weighted',
                 'jarvis', 'sierra_lite', 'burkes',
                 'bayer4', 'bayer8', 'pil_fs', 'none']
SHOW_ALL = False
ALGOS = list(DITHER_ALGORITHMS.keys()) if SHOW_ALL else DEFAULT_ALGOS

N_PHOTOS = 2  # one image cycle per photo
SEED = 11

In [ ]:
client = immich_client()
pool_assets = fetch_pool(client, pool_size=20, seed=SEED)

with silenced():
    sources = []
    for asset in pool_assets[:N_PHOTOS]:
        img = download_image(client, asset)
        sources.append((asset, _crop_center(img, EPD_WIDTH, EPD_HEIGHT)))

for asset, _ in sources:
    print(asset.get('originalFileName') or asset['id'])

# Render the 6-colour palette as a tiny banner so the colour budget is visible.
swatch_h = 60
swatch_w = 60
palette_strip = np.zeros((swatch_h, swatch_w * len(PALETTE_RGB), 3), dtype=np.uint8)
for i, rgb in enumerate(PALETTE_RGB):
    palette_strip[:, i * swatch_w:(i + 1) * swatch_w] = rgb
Image.fromarray(palette_strip)

In [ ]:
results = []  # list of dict per (photo, algo)
for asset, source in sources:
    name = asset.get('originalFileName') or asset['id']
    print(f'[{name}]')
    photo_results = []
    for algo in ALGOS:
        info = DITHER_ALGORITHMS[algo]
        t0 = time.perf_counter()
        out = apply_dithering(source, algo)
        dt = time.perf_counter() - t0
        photo_results.append({'algo': algo, 'name': info['name'], 'image': out, 'duration': dt})
        print(f'  {info["name"]:32s}  {dt:6.2f}s')
    results.append({'asset': asset, 'source': source, 'algos': photo_results})

In [ ]:
# One grid per photo: original + every algorithm.
import matplotlib.pyplot as plt
for entry in results:
    panels = [entry['source']] + [r['image'] for r in entry['algos']]
    titles = ['original (cropped)'] + [f"{r['name']}\n{r['duration']:.2f}s" for r in entry['algos']]
    cols = 4
    rows = (len(panels) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5.0 * cols, 3.2 * rows))
    axes = np.atleast_2d(axes)
    name = entry['asset'].get('originalFileName') or entry['asset']['id']
    fig.suptitle(name, fontsize=12)
    for k in range(rows * cols):
        ax = axes[k // cols][k % cols]
        if k < len(panels):
            ax.imshow(panels[k])
            ax.set_title(titles[k], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-algorithm summary across both photos: mean runtime + a single representative panel.
from collections import defaultdict
import matplotlib.pyplot as plt

agg = defaultdict(list)
for entry in results:
    for r in entry['algos']:
        agg[r['algo']].append(r['duration'])

print(f"{'algorithm':32s}  {'avg time':>9s}  description")
for algo in ALGOS:
    info = DITHER_ALGORITHMS[algo]
    avg = np.mean(agg[algo])
    print(f"{info['name']:32s}  {avg:>8.2f}s  {info['description']}")

## Picking an algorithm

- **Photographs** — `atkinson_fast` (production), `atkinson_weighted`, or
  `floyd_steinberg_weighted`. Atkinson loses some detail (only diffuses 6/8 of the error)
  but gives cleaner edges; FS preserves detail at the cost of more visible noise.
- **Graphics / illustrations / posters** — `bayer4` or `bayer8`. The pattern is regular
  (no "wormy" artifacts) and well-suited to large flat regions.
- **Speed-critical paths** — `atkinson_fast` (numba-JIT). Pure-Python error-diffusion
  algorithms are ~150× slower than the JIT'd version on this resolution.

The `none` row (PIL nearest-colour) shows what happens with no dithering at all — useful as
a baseline to confirm the dithering is buying you something.

**To compare more algorithms** set `SHOW_ALL = True` in the setup cell. Expect several
minutes of CPU time per photo for the slow pure-Python implementations.